In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: True
GPU name: Tesla T4


In [2]:
!pip install -q transformers datasets accelerate sentencepiece


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

train_df = pd.read_json("/content/drive/MyDrive/rewriter_datasets/t5_rewriter_train_split.jsonl", lines=True)
val_df = pd.read_json("/content/drive/MyDrive/rewriter_datasets/t5_rewriter_val_split.jsonl", lines=True)

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))

print("\nSample:")
print(train_df.iloc[0]["input_text"])
print("→", train_df.iloc[0]["target_text"])

Train rows: 7018
Val rows: 780

Sample:
rewrite query: What are some of the surgical approaches that have been suggested for accessing the atrium of the lateral ventricle?

→ accessing atrium lateral ventricle surgery


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-small"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded on device:", device)

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model...
Model loaded on device: cuda


In [6]:
# Sanity test: run a single forward pass

test_input = "rewrite query: What are the advantages of minimally invasive surgery for intracerebral hematomas?"

inputs = tokenizer(
    test_input,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=32,
        num_beams=4
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Input :", test_input)
print("Output:", decoded)

Input : rewrite query: What are the advantages of minimally invasive surgery for intracerebral hematomas?
Output: The advantages of minimally invasive surgery for intracerebral hematomas are: minimally invasive surgery for intracerebra


In [7]:
from datasets import load_dataset

TRAIN_PATH = "/content/drive/MyDrive/rewriter_datasets/t5_rewriter_train_split.jsonl"
VAL_PATH   = "/content/drive/MyDrive/rewriter_datasets/t5_rewriter_val_split.jsonl"

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_PATH,
        "validation": VAL_PATH
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7018
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 780
    })
})


In [8]:
def tokenize_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        batch["target_text"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [9]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

print(tokenized_datasets)

Map:   0%|          | 0/7018 [00:00<?, ? examples/s]

Map:   0%|          | 0/780 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7018
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 780
    })
})


In [10]:
sample = tokenized_datasets["train"][0]

print("input_ids shape:", len(sample["input_ids"]))
print("labels shape:", len(sample["labels"]))
print("First 10 input_ids:", sample["input_ids"][:10])
print("First 10 labels:", sample["labels"][:10])

input_ids shape: 32
labels shape: 11
First 10 input_ids: [3, 60, 17504, 11417, 10, 363, 33, 128, 13, 8]
First 10 labels: [592, 53, 44, 11879, 3, 12088, 9370, 2234, 109, 3730]


In [11]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

In [15]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Evaluate every epoch
    save_strategy="epoch",       # <--- ADD THIS: Save every epoch to match eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,          # Will now keep the best 2 epoch checkpoints
    load_best_model_at_end=True,
    fp16=False,                  # Keep False for T4 stability
    report_to="none"
)

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized successfully.")


Trainer initialized successfully.


/tmp/ipython-input-2465420648.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.210000,0.911717
2,1.119600,0.858974
3,1.104800,0.846783


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1317, training_loss=1.2043166685575053, metrics={'train_runtime': 215.7728, 'train_samples_per_second': 97.575, 'train_steps_per_second': 6.104, 'total_flos': 374259307425792.0, 'train_loss': 1.2043166685575053, 'epoch': 3.0})

In [18]:
def rewrite_question(question):
    text = "rewrite query: " + question
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print(
    rewrite_question(
        "What are the advantages of minimally invasive surgical techniques for intracerebral hematomas compared to craniotomy?"
    )
)

minimally invasive surgical techniques intracerebral hematomas


In [19]:
SAVE_DIR = "/content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved to:", SAVE_DIR)

Model saved to: /content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter


In [21]:
!zip -r flan_t5_neuro_rewriter.zip /content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter


  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/ (stored 0%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/config.json (deflated 62%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/generation_config.json (deflated 27%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/tokenizer_config.json (deflated 95%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/special_tokens_map.json (deflated 85%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/spiece.model (deflated 48%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/tokenizer.json (deflated 74%)
  adding: content/drive/MyDrive/rewriter_model/flan_t5_neuro_rewriter/training_args.bin (deflated 53%)


In [22]:
from google.colab import files
files.download("flan_t5_neuro_rewriter.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>